# Time-restriction sensitivity

Same incident BASE / CLIN / SOC (+NMR) ridge Cox models as `primary_incident_ckm.ipynb`, with one change: incident follow-up must exceed a lag window. Prevalent cases are already dropped by `prep_incident_time_to_event_df()`.

- `exclude_6mo`: `time_col > 0.5` (times are in years)
- `exclude_1yr`: `time_col > 1`

No PRS specs. `fit_full` stays `FALSE`. Fold models and absolute-risk objects are not saved. Delta C-index is not computed.

Outputs stay out of `outputs/main`:

```
outputs/supp/time_restriction/exclude_6mo/<tag>_<spec>_oof.rds
outputs/supp/time_restriction/exclude_1yr/<tag>_<spec>_oof.rds
outputs/tables/CINDEX_STATS_TIME_RESTRICT_6mo.csv
outputs/tables/CINDEX_STATS_TIME_RESTRICT_1yr.csv
outputs/figures/TIME_RESTRICT_6mo.png
outputs/figures/TIME_RESTRICT_1yr.png
```

Existing `.rds` files are skipped (`skip_existing = TRUE`).


In [ ]:
suppressPackageStartupMessages({
  library(ggpubr)
  library(patchwork)
  library(tidyverse)
  library(survival)
  library(glmnet)
  library(here)
})


# Load data


In [ ]:
source(here("helpers", "prep_incident_time_to_event_df.r"))
source(here("helpers", "summarize_time_to_event.r"))
source(here("helpers", "cv_glmnet_oof.r"))
source(here("helpers", "fit_full_cox.r"))

meta <- read.csv(here("data", "meta_clin.csv"), row.names = 1) %>%
  dplyr::mutate(StudyID = as.character(StudyID))
assay <- read.csv(here("data", "assay.csv"), row.names = 1) %>%
  dplyr::mutate(StudyID = as.character(StudyID))
nmr_names <- assay %>% select(-StudyID) %>% names()

cat("meta dimensions:", dim(meta), "\n")
cat("assay dimensions:", dim(assay), "\n")


# Configuration


In [ ]:
cv_defaults <- list(
  outer_nfolds   = 5,
  inner_nfolds   = 10,
  nlambda        = 100,
  seed           = 1,
  standardize    = FALSE,
  parallel_outer = TRUE,
  n_cores        = 5L,
  alpha          = 0
)

common_blocks <- list(
  base = c("age_at_sample", "sex"),
  clin = c("BMI_interp", "SBP_interp"),
  nmr  = nmr_names
)

common_scale_blocks <- list(
  base = c("age_at_sample"),
  clin = c("BMI_interp", "SBP_interp"),
  nmr  = nmr_names
)

model_specs <- list(
  base       = c("base"),
  base_nmr   = c("base", "nmr"),
  clin       = c("base", "clin"),
  clin_nmr   = c("base", "clin", "nmr"),
  marker     = c("base", "clin", "marker"),
  marker_nmr = c("base", "clin", "marker", "nmr")
)

# Years. 0.5 = 6 months.
time_windows <- list(
  exclude_6mo = 0.5,
  exclude_1yr = 1
)

TIME_RESTRICT_ROOT <- here("outputs/supp/time_restriction")
FIG_ROOT <- here("outputs/figures")
for (w in names(time_windows)) {
  dir.create(file.path(TIME_RESTRICT_ROOT, w, "c_index_summaries"), recursive = TRUE, showWarnings = FALSE)
}
dir.create(FIG_ROOT, recursive = TRUE, showWarnings = FALSE)


In [ ]:
resolve_blocks <- function(block_names, block_map) {
  missing_blocks <- setdiff(block_names, names(block_map))
  if (length(missing_blocks) > 0) {
    stop("Missing block definitions: ", paste(missing_blocks, collapse = ", "))
  }
  unique(unlist(block_map[block_names], use.names = FALSE))
}

make_penalty_factor <- function(predictors, unpenalized = character()) {
  pf <- rep(1, length(predictors))
  names(pf) <- predictors
  pf[intersect(unpenalized, predictors)] <- 0.1
  pf
}

make_model_maps <- function(marker_predictors, marker_scale = character()) {
  list(
    blocks = c(common_blocks, list(marker = marker_predictors)),
    scale_blocks = c(common_scale_blocks, list(marker = marker_scale))
  )
}

get_required_vars <- function(maps,
                              specs = c("base", "clin", "marker"),
                              extra_vars = c("time", "status"),
                              exclude_blocks = c("nmr")) {
  used_blocks <- unique(unlist(model_specs[specs], use.names = FALSE))
  used_blocks <- setdiff(used_blocks, exclude_blocks)
  unique(c(extra_vars, resolve_blocks(used_blocks, maps$blocks)))
}

filter_complete_cases_for_specs <- function(df, maps,
                                            specs = c("base", "clin", "marker"),
                                            extra_vars = c("time", "status"),
                                            exclude_blocks = c("nmr")) {
  vars_needed <- get_required_vars(
    maps = maps,
    specs = specs,
    extra_vars = extra_vars,
    exclude_blocks = exclude_blocks
  )
  df %>% filter(if_all(all_of(vars_needed), ~ !is.na(.)))
}

print_cv_summary <- function(results) {
  cat(sprintf("\nOOF C-index: %.4f (SE: %.4f)\n", results$c_index, results$se))
  cat(sprintf("95%% CI: [%.4f, %.4f]\n", results$ci_lower, results$ci_upper))
}

run_cv_spec <- function(df, spec_name, maps,
                        unpenalized = character(),
                        output_path = NULL,
                        model_engine = "glmnet",
                        fit_full = FALSE,
                        keep_fold_models = FALSE) {
  if (!spec_name %in% names(model_specs)) {
    stop("Unknown spec_name: ", spec_name)
  }

  block_names   <- model_specs[[spec_name]]
  predictors    <- resolve_blocks(block_names, maps$blocks)
  vars_to_scale <- resolve_blocks(block_names, maps$scale_blocks)

  missing_predictors <- setdiff(c("time", "status", predictors), names(df))
  if (length(missing_predictors) > 0) {
    stop(sprintf("Spec '%s' requested variables not found in df: %s",
                 spec_name, paste(missing_predictors, collapse = ", ")))
  }
  missing_scale <- setdiff(vars_to_scale, names(df))
  if (length(missing_scale) > 0) {
    stop(sprintf("Spec '%s' vars_to_scale not found in df: %s",
                 spec_name, paste(missing_scale, collapse = ", ")))
  }

  df_model <- df %>%
    dplyr::select(dplyr::all_of(c("StudyID", "time", "status", predictors))) %>%
    dplyr::filter(dplyr::if_all(dplyr::all_of(c("time", "status", predictors)), ~ !is.na(.)))

  unpenalized <- intersect(unpenalized, predictors)
  penalty.factor <- make_penalty_factor(predictors, unpenalized)

  args <- c(
    list(
      df = df_model,
      predictors = predictors,
      vars_to_scale = vars_to_scale,
      penalty.factor = penalty.factor,
      model_engine = model_engine
    ),
    cv_defaults
  )
  # NMR design matrices are too large to copy onto 5 PSOCK workers.
  if ("nmr" %in% block_names) {
    args$parallel_outer <- FALSE
  }
  results <- do.call(cv_glmnet_cox_oof, args)
  print_cv_summary(results)

  if (!isTRUE(keep_fold_models)) {
    results$fold_models <- NULL
  }
  invisible(gc(verbose = FALSE))

  if (isTRUE(fit_full)) {
    full_args <- c(
      list(
        df             = df_model,
        predictors     = predictors,
        vars_to_scale  = vars_to_scale,
        penalty.factor = penalty.factor,
        model_engine   = model_engine
      ),
      cv_defaults[setdiff(names(cv_defaults), c("outer_nfolds", "parallel_outer", "n_cores"))]
    )
    results$full_model <- do.call(fit_full_cox, full_args)
  }

  if (!is.null(output_path)) {
    saveRDS(results, output_path)
  }
  invisible(results)
}


In [ ]:
# Marker maps match primary_incident_ckm.ipynb. Columns are already on meta_clin.
t2d_maps <- make_model_maps(
  marker_predictors = c("M_a1c", "A1C_obs", "T_a1c"),
  marker_scale      = c("A1C_obs", "T_a1c")
)

ckd_maps <- make_model_maps(
  marker_predictors = c("M_egfr", "EGFR_obs", "T_egfr"),
  marker_scale      = c("EGFR_obs", "T_egfr")
)

nafld_maps <- make_model_maps(
  marker_predictors = c("M_fib4", "FIB4_obs", "T_fib4"),
  marker_scale      = c("FIB4_obs", "T_fib4")
)

copd_maps <- make_model_maps(
  marker_predictors = c("M_smoke", "smk_current", "smk_former", "M_ldl", "LDL_obs", "T_ldl"),
  marker_scale      = c("LDL_obs", "T_ldl")
)

chd_complete_maps <- make_model_maps(
  marker_predictors = c(
    "M_tot_choles", "TOT_CHOLES_obs", "T_tot_choles",
    "M_hdl_choles", "HDL_CHOLES_obs", "T_hdl_choles",
    "M_egfr", "EGFR_obs", "T_egfr",
    "M_smoke", "smk_current", "smk_former",
    "bp_treatment", "statin_treatment", "diabetes_at_baseline"
  ),
  marker_scale = c(
    "TOT_CHOLES_obs", "T_tot_choles",
    "HDL_CHOLES_obs", "T_hdl_choles",
    "EGFR_obs", "T_egfr"
  )
)

afib_complete_maps <- make_model_maps(
  marker_predictors = c(
    "M_smoke_af", "smoking_current_af",
    "height_interp", "weight_interp", "DBP_interp",
    "bp_treatment", "hf_at_baseline", "mi_at_baseline",
    "diabetes_at_baseline", "race"
  ),
  marker_scale = c("height_interp", "weight_interp", "DBP_interp")
)

endpoint_config <- list(
  t2d = list(
    label = "T2D", plot_label = "Type 2 Diabetes", tag = "t2d_a1c",
    status_col = "t2d_status", time_col = "t2d_time",
    maps = t2d_maps
  ),
  ckd = list(
    label = "CKD", plot_label = "Chronic Kidney Disease", tag = "ckd_egfr",
    status_col = "ckd_status", time_col = "ckd_time",
    maps = ckd_maps
  ),
  masld = list(
    label = "MASLD", plot_label = "MASLD (Liver Disease)", tag = "nafld_fib4",
    status_col = "nafld_status", time_col = "nafld_time",
    maps = nafld_maps
  ),
  copd = list(
    label = "COPD", plot_label = "Chronic Obstructive Pulmonary Disease", tag = "copd_smoking_ldl",
    status_col = "copd_status", time_col = "copd_time",
    maps = copd_maps
  ),
  chd_revasc = list(
    label = "CHD", plot_label = "Coronary Heart Disease", tag = "chd_revasc_prevent_ingredients",
    status_col = "chd_minerva_status", time_col = "chd_minerva_time",
    maps = chd_complete_maps
  ),
  hf_cm = list(
    label = "Heart Failure", plot_label = "Heart Failure", tag = "hf_cm_prevent_ingredients",
    status_col = "hf_cm_status", time_col = "hf_cm_time",
    maps = chd_complete_maps
  ),
  acute_mi = list(
    label = "Acute MI", plot_label = "Acute MI", tag = "acute_mi_prevent_ingredients",
    status_col = "acute_mi_status", time_col = "acute_mi_time",
    maps = chd_complete_maps
  ),
  tia_stroke = list(
    label = "TIA/Stroke", plot_label = "TIA/Stroke", tag = "tia_stroke_prevent_ingredients",
    status_col = "tia_stroke_status", time_col = "tia_stroke_time",
    maps = chd_complete_maps
  ),
  total_cvd = list(
    label = "Total CVD", plot_label = "Total CVD", tag = "total_cvd_prevent_ingredients",
    status_col = "total_cvd_status", time_col = "total_cvd_time",
    maps = chd_complete_maps
  ),
  afib = list(
    label = "Afib", plot_label = "Atrial fibrillation", tag = "afib_chargeaf_ingredients",
    status_col = "afib_status", time_col = "afib_time",
    maps = afib_complete_maps
  )
)


In [ ]:
oof_path <- function(window, tag, spec) {
  file.path(TIME_RESTRICT_ROOT, window, paste0(tag, "_", spec, "_oof.rds"))
}

prepare_endpoint_df <- function(nm, min_years) {
  cfg <- endpoint_config[[nm]]
  if (is.null(cfg)) stop("Unknown endpoint: ", nm)

  time_filter <- paste(cfg$time_col, ">", min_years)
  message("Preparing ", nm, " (", cfg$label, ") with time_filter: ", time_filter)

  meta_use <- prep_incident_time_to_event_df(
    meta,
    status_col  = cfg$status_col,
    time_col    = cfg$time_col,
    time_filter = time_filter
  )
  print(summarize_time_to_event(meta_use))

  meta_use <- filter_complete_cases_for_specs(
    df = meta_use,
    maps = cfg$maps,
    specs = c("base", "clin", "marker")
  )
  meta_use_nmr <- meta_use %>% inner_join(assay, by = "StudyID")

  cat("analytical n:", nrow(meta_use_nmr),
      " events:", sum(meta_use_nmr$status), "\n")

  list(df = meta_use_nmr, maps = cfg$maps, cfg = cfg)
}

run_endpoint_window <- function(nm, window,
                                specs = names(model_specs),
                                skip_existing = TRUE,
                                fit_full = FALSE) {
  if (!window %in% names(time_windows)) {
    stop("Unknown window: ", window, ". Use one of: ",
         paste(names(time_windows), collapse = ", "))
  }
  cfg <- endpoint_config[[nm]]
  if (is.null(cfg)) stop("Unknown endpoint: ", nm)

  prepared <- prepare_endpoint_df(nm, time_windows[[window]])

  for (sp in specs) {
    path <- oof_path(window, cfg$tag, sp)
    if (skip_existing && file.exists(path)) {
      message("Skipping existing OOF: ", window, "/", basename(path))
      next
    }
    cat("\n==============================\n", window, "::", cfg$tag, "::", sp,
        "\n==============================\n")
    results <- run_cv_spec(
      df           = prepared$df,
      spec_name    = sp,
      maps         = prepared$maps,
      unpenalized  = c("age_at_sample", "sex"),
      model_engine = "glmnet",
      output_path  = path,
      fit_full     = fit_full,
      keep_fold_models = FALSE
    )
    rm(results)
    cleanup_all_clusters()
    gc(verbose = FALSE)
  }

  invisible(list(
    endpoint = nm,
    window   = window,
    tag      = cfg$tag,
    n        = nrow(prepared$df),
    n_events = sum(prepared$df$status)
  ))
}

collect_time_restrict_summary <- function() {
  rows <- list()
  for (window in names(time_windows)) {
    for (nm in names(endpoint_config)) {
      cfg <- endpoint_config[[nm]]
      for (sp in names(model_specs)) {
        path <- oof_path(window, cfg$tag, sp)
        if (!file.exists(path)) next
        res <- tryCatch(readRDS(path), error = function(e) {
          warning("Could not read ", path, ": ", conditionMessage(e))
          NULL
        })
        if (is.null(res)) next
        oof <- res$oof_predictions
        rows[[length(rows) + 1L]] <- data.frame(
          window   = window,
          endpoint = nm,
          label    = cfg$label,
          tag      = cfg$tag,
          spec     = sp,
          n        = nrow(oof),
          events   = sum(oof$status),
          c_index  = res$c_index,
          se       = res$se,
          ci_lower = res$ci_lower,
          ci_upper = res$ci_upper,
          stringsAsFactors = FALSE
        )
        rm(res)
      }
    }
  }
  dplyr::bind_rows(rows)
}


## How to run

Run the cells above, then the full grid. `skip_existing = TRUE` resumes from the first missing `.rds`. Set it to `FALSE` to refit.

```r
run_endpoint_window("t2d", "exclude_6mo")
run_endpoint_window("ckd", "exclude_1yr", specs = c("marker", "marker_nmr"))
```


In [ ]:
# Smoke test: T2D, 6-month exclusion, all six specs.
# t2d_6mo <- run_endpoint_window("t2d", "exclude_6mo", skip_existing = TRUE)
# t2d_6mo


In [ ]:
# Full grid: 10 endpoints x 2 windows x 6 specs.
failed <- character()
for (window in names(time_windows)) {
  for (nm in names(endpoint_config)) {
    message("\n#### ", window, " / ", nm, " (", endpoint_config[[nm]]$label, ") ####")
    tryCatch(
      run_endpoint_window(nm, window, skip_existing = TRUE),
      error = function(e) {
        failed <<- c(failed, paste0(window, "/", nm))
        warning("Failed ", window, "/", nm, ": ", conditionMessage(e))
        NULL
      }
    )
  }
}
if (length(failed) > 0) {
  warning("Failed endpoint-windows: ", paste(failed, collapse = ", "))
} else {
  message("All endpoint x window combinations finished (or were already on disk).")
}

## Absolute C-index

Reads saved OOF files and writes one dumbbell figure per lag window. COPD is fit and included in the cohort summary, and omitted from the figures and wide tables.


In [ ]:
source(here("helpers", "plot_oof_cindex_primary.r"))

FIG_PANEL_ORDER <- c(
  "t2d", "ckd", "masld",
  "hf_cm", "acute_mi", "chd_revasc",
  "tia_stroke", "total_cvd", "afib"
)

WINDOW_TITLES <- c(
  exclude_6mo = "Incident disease < 6 months excluded",
  exclude_1yr = "Incident disease < 1 year excluded"
)

WINDOW_FILES <- c(
  exclude_6mo = "TIME_RESTRICT_6mo",
  exclude_1yr = "TIME_RESTRICT_1yr"
)

SPEC_MAP <- data.frame(
  spec  = c("base", "base_nmr", "clin", "clin_nmr", "marker", "marker_nmr"),
  model = c("base", "base", "clin", "clin", "marker", "marker"),
  type  = c("SOC", "NMR", "SOC", "NMR", "SOC", "NMR"),
  stringsAsFactors = FALSE
)

endpoint_cindex_ready <- function(df_ep) {
  nrow(df_ep) > 0 &&
    identical(sort(unique(df_ep$spec)), sort(names(model_specs)))
}

write_cindex_csvs <- function(summary_df) {
  if (is.null(summary_df) || nrow(summary_df) == 0L) {
    message("No OOF summaries on disk yet.")
    return(invisible(character()))
  }
  written <- character()
  for (w in unique(summary_df$window)) {
    df_w <- summary_df[summary_df$window == w, , drop = FALSE]
    for (nm in unique(df_w$endpoint)) {
      df_ep <- df_w[df_w$endpoint == nm, , drop = FALSE]
      if (!endpoint_cindex_ready(df_ep)) {
        message("Skipping CSV (incomplete specs): ", w, "/", nm)
        next
      }
      tag <- unique(df_ep$tag)
      path <- file.path(TIME_RESTRICT_ROOT, w, "c_index_summaries", paste0(tag, "_cindex_ci.csv"))
      write.csv(prep_cindex_dumbbell_from_specs(df_ep), path, row.names = FALSE)
      written <- c(written, path)
      message("Wrote ", path)
    }
  }
  invisible(written)
}

plot_cindex_window <- function(window, summary_df, xlim = c(0.5, 0.85)) {
  df_w <- summary_df[summary_df$window == window, , drop = FALSE]
  plots <- list()
  missing <- character()

  for (nm in FIG_PANEL_ORDER) {
    df_ep <- df_w[df_w$endpoint == nm, , drop = FALSE]
    if (!endpoint_cindex_ready(df_ep)) {
      missing <- c(missing, nm)
      next
    }
    plots[[nm]] <- plot_oof_cindex(
      df = prep_cindex_dumbbell_from_specs(df_ep),
      title = endpoint_config[[nm]]$plot_label,
      xlim = xlim,
      marker_label = "SOC",
      type_labels = c("SOC" = "No NMR", "NMR" = "NMR")
    )
  }

  if (length(missing) > 0) {
    message("Window ", window, " missing panels: ", paste(missing, collapse = ", "))
  }
  if (length(plots) == 0L) {
    warning("No complete endpoints to plot for window ", window)
    return(invisible(NULL))
  }

  hide_legend <- function(p) p + theme(legend.position = "none")
  show_bottom <- function(p) p + theme(legend.position = "bottom")

  if (length(plots) == 9L && identical(names(plots), FIG_PANEL_ORDER)) {
    p_all <-
      (hide_legend(plots$t2d) | hide_legend(plots$ckd) | hide_legend(plots$masld)) /
      (hide_legend(plots$hf_cm) | hide_legend(plots$acute_mi) | hide_legend(plots$chd_revasc)) /
      (hide_legend(plots$tia_stroke) | show_bottom(plots$total_cvd) | hide_legend(plots$afib))
  } else {
    plots_adj <- lapply(names(plots), function(nm) {
      if (identical(nm, "total_cvd")) show_bottom(plots[[nm]]) else hide_legend(plots[[nm]])
    })
    p_all <- patchwork::wrap_plots(plots_adj, ncol = 3)
  }

  p_all + patchwork::plot_annotation(
    title = unname(WINDOW_TITLES[[window]]),
    theme = ggplot2::theme(plot.title = ggplot2::element_text(hjust = 0.5))
  )
}

wide_cindex_table <- function(summary_df) {
  endpoint_order <- c(
    "T2D", "CKD", "MASLD",
    "Heart Failure", "Acute MI", "CHD",
    "TIA/Stroke", "Total CVD", "Afib"
  )
  col_order <- c("endpoint", "BASE", "BASE+NMR", "CLIN", "CLIN+NMR", "SOC", "SOC+NMR")

  summary_df %>%
    filter(endpoint %in% FIG_PANEL_ORDER) %>%
    left_join(SPEC_MAP, by = "spec") %>%
    mutate(
      endpoint = factor(label, levels = endpoint_order),
      col = case_when(
        model == "base"   & type == "SOC" ~ "BASE",
        model == "base"   & type == "NMR" ~ "BASE+NMR",
        model == "clin"   & type == "SOC" ~ "CLIN",
        model == "clin"   & type == "NMR" ~ "CLIN+NMR",
        model == "marker" & type == "SOC" ~ "SOC",
        model == "marker" & type == "NMR" ~ "SOC+NMR",
        TRUE ~ NA_character_
      ),
      value = sprintf("%.3f (%.3f, %.3f)", c_index, ci_lower, ci_upper)
    ) %>%
    filter(!is.na(col)) %>%
    select(endpoint, col, value) %>%
    tidyr::pivot_wider(names_from = col, values_from = value) %>%
    arrange(endpoint) %>%
    select(any_of(col_order))
}


In [ ]:
time_restrict_summary <- collect_time_restrict_summary()
print(time_restrict_summary)

write.csv(
  time_restrict_summary,
  file.path(TIME_RESTRICT_ROOT, "cohort_cindex_summary.csv"),
  row.names = FALSE
)
write_cindex_csvs(time_restrict_summary)

options(repr.plot.height = 4.25, repr.plot.width = 7)

for (window in names(time_windows)) {
  df_w <- time_restrict_summary %>% filter(window == !!window)
  cindex_table <- wide_cindex_table(df_w)
  print(cindex_table)
  write.csv(
    cindex_table,
    here("outputs/tables", paste0("CINDEX_STATS_", WINDOW_FILES[[window]], ".csv")),
    row.names = FALSE
  )

  p_window <- plot_cindex_window(window, time_restrict_summary)
  print(p_window)
  if (!is.null(p_window)) {
    ggsave(
      file.path(FIG_ROOT, paste0(WINDOW_FILES[[window]], ".png")),
      plot = p_window,
      width = 7,
      height = 4.25,
      dpi = 600
    )
  }
}